# AWS Workshop · Train a text emotion classifier

In this notebook, you'll teach DistilBERT to classify short English texts into six emotions. You'll inspect labeled examples, turn text into numbers the model can process, and see how well it predicts emotions as it trains.

See the [workshop README](README.md) for JupyterLab setup and the optional managed SageMaker training exercise.

## Train in the notebook

Run one code cell at a time with **Shift+Enter**. Start at the top because later cells reuse variables such as the dataset and tokenizer from earlier cells. The installation step calls for a kernel restart. After that restart, continue with the imports cell.

### Install libraries

`%pip` installs Python packages into the current notebook kernel's environment. The `-q` flag keeps routine installation output short.

- `tf-keras` provides the legacy Keras package for TensorFlow compatibility. The training cells below use PyTorch.
- `transformers` provides the pretrained model, tokenizer, and training loop.
- `datasets` loads and prepares the labeled examples.
- `accelerate` helps the training loop use the available hardware.
- `sagemaker` provides the AWS SDK used in the managed training exercise in the README.

Once installation finishes, choose **Kernel → Restart Kernel** so the notebook loads the installed libraries on the next import. Continue with the next cell after the restart.

In [ ]:
%pip install -q tf-keras
%pip install -q "transformers>=4.44,<5" "datasets>=2.20" "accelerate>=0.26" "sagemaker>=2.200,<3"

### Import libraries and check the hardware

The cell prints the versions of PyTorch, Transformers, and Datasets that Python loaded. If an import fails, check that the installer completed and that you restarted the kernel.

`torch.cuda.is_available()` checks whether PyTorch can use a CUDA GPU. The workshop's CPU instance should print `GPU available? False` alongside the three version numbers. PyTorch can run this exercise on the CPU.

In [ ]:
import torch, transformers, datasets
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("torch:", torch.__version__)
print("GPU available?", torch.cuda.is_available())

### Load the dataset from Hugging Face

Each row contains a short text and an integer representing its emotion. The code looks up the emotion names in the dataset metadata, then uses those names to print five readable examples.

The dataset has 16,000 training rows, 2,000 validation rows, and 2,000 test rows. Training rows are the examples the model learns from. This workshop takes evaluation examples from the test split and leaves the validation split unused. Read a few of the printed texts and see whether you'd pick the same emotion.

In [ ]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion")
print(ds)

labels = ds["train"].features["label"].names
print("Labels:", labels)

print("\nA few examples:")
for i in range(5):
    row = ds["train"][i]
    print(f"  {labels[row['label']]:>9} | {row['text']}")

### Sample the training and evaluation data

Using fewer examples keeps the exercise manageable on a CPU. The cell shuffles each split separately, then selects 2,000 training examples and 500 test examples. The training loop learns from the first sample and measures accuracy on the second.

The fixed seed of `42` makes the shuffle repeatable for the same dataset. Check that the two printed counts are `2000` and `500`.

In [ ]:
train_ds = ds["train"].shuffle(seed=42).select(range(2000))
test_ds  = ds["test"].shuffle(seed=42).select(range(500))

print("Training examples:", len(train_ds))
print("Test examples:", len(test_ds))

### See how the tokenizer represents text

DistilBERT processes numbers. Its tokenizer breaks a sentence into tokens, which can be words or pieces of words, and maps each token to an integer ID in the model's vocabulary. Loading the tokenizer from the same checkpoint as the model keeps those IDs aligned.

Compare the two printed lists position by position. One shows the IDs and the other shows the corresponding tokens. `[CLS]` marks the start and supplies a position the classifier uses to represent the sentence. `[SEP]` marks the end of this single sentence.

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

example = "i feel so happy today"
encoded = tokenizer(example)
print("Token IDs:", encoded["input_ids"])
print("Tokens:   ", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

### Tokenize the whole dataset

The `map` calls apply the tokenizer to both samples. With `batched=True`, the tokenizer receives a group of texts at once. Each output sequence has 64 positions, including special tokens. Truncation cuts longer sequences to fit, and padding fills the unused positions in shorter ones. The tokenizer also returns an attention mask to tell the model which positions are padding.

The original emotion labels stay attached to their rows so the trainer can compare predictions with the known answers. Continue when you see `Done tokenizing!`.

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

train_tok = train_ds.map(tokenize, batched=True)
test_tok  = test_ds.map(tokenize, batched=True)
print("Done tokenizing!")

### Load the pretrained model

DistilBERT starts with weights learned during earlier training. The cell adds a classification layer that produces one score for each of the six emotions. The `id2label` and `label2id` dictionaries map between score positions and emotion names.

Expect around 67 million parameters and a warning that some weights were newly initialized. Those weights belong to the classification layer. The training step will adapt the model to this emotion classification task, a process called **fine-tuning**.

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)
print(f"Model loaded with {model.num_parameters():,} parameters")

### Test before training

The four example sentences give you a baseline. For each one, `predict` tokenizes the text, moves the inputs onto the model's device, and runs a prediction without calculating gradients.

The model outputs six raw scores called **logits**. Softmax turns them into scores that add up to one, and `argmax` picks the largest. The printed percentage is the model's score for the label it picked. Expect weak or repetitive guesses because the classification layer has just been initialized. Keep this output visible for comparison after training.

In [ ]:
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    top = int(probs.argmax())
    return f"{id2label[top]} ({probs[top]:.0%} sure)"

test_sentences = [
    "i just got accepted into my dream internship",
    "i miss my family so much it hurts",
    "why would you lie to me like that",
    "i heard a noise downstairs and i am home alone",
]

print("BEFORE training:")
for s in test_sentences:
    print(f"  {s!r} -> {predict(s)}")

### Train the model and measure accuracy

The **Trainer** feeds the model up to 16 labeled examples at a time. It calculates a **loss** from the model's predictions and the known labels, then updates the model's weights. The learning rate controls the size of those updates. An **epoch** is one pass through the training sample. With 2,000 examples, two epochs produce 250 training steps on a single device.

Every 25 steps, the trainer logs progress. At the end of each epoch, it evaluates on the 500 sampled test examples. The `compute_metrics` function selects the highest-scoring label for each row and reports the fraction that matches the known labels.

Watch the loss across several updates and compare accuracy across epochs. If training is too slow, interrupt the cell, change the training sample to 1,000 rows, and rerun from the sampling cell through training to start with a fresh model.

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == true_labels).mean())}

training_args = TrainingArguments(
    output_dir="emotion-model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

trainer.train()

### Read the training results

Start with the accuracy reported after each epoch. An accuracy of 70% means the model chose the correct label for about 350 of the 500 evaluation examples. A uniformly random guess across six labels would average about 17%.

To compare individual predictions, run `predict` on the same example sentences after training without reloading the model. You can also try a sentence of your own. The classifier always chooses from the six labels, even for neutral text or sentences that express mixed feelings.